# 04 Options Surface

Purpose: inspect option chain features, implied volatility, Greeks, smile and surface grids.

Data source: `data/processed/options_chain_features.parquet` from public/delayed MOEX ISS options fields.


## Methodology Summary

The pipeline parses option type, strike, expiration and underlying where available. Black-Scholes is used as a transparent approximation; IV is NaN when inputs are invalid or the solver fails.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from russian_markets_lab.data.io import read_processed_dataset, read_processed_metadata

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

def load_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    df = read_processed_dataset(name)
    metadata = read_processed_metadata(name)
    print(f'{name}: {len(df)} rows, {len(df.columns)} columns')
    if metadata:
        print('generated_at:', metadata.get('generated_at'))
        print('source:', metadata.get('source'))
    else:
        print('metadata: missing')
    return df, metadata

def show_missing(name: str) -> None:
    print(f'No processed data found for {name}. Run the corresponding CLI pipeline first.')


In [ ]:
chain, metadata = load_dataset('options_chain_features')
if chain.empty:
    show_missing('options_chain_features')
else:
    display(chain.head(30))


## Implied Volatility Smile


In [ ]:
if not chain.empty and {'expiration', 'strike', 'implied_volatility'}.issubset(chain.columns):
    valid = chain.dropna(subset=['expiration', 'strike', 'implied_volatility'])
    if not valid.empty:
        expiry = valid['expiration'].astype(str).sort_values().iloc[0]
        smile = valid[valid['expiration'].astype(str) == expiry]
        fig = px.scatter(smile, x='strike', y='implied_volatility', color='option_type' if 'option_type' in smile.columns else None, title=f'IV smile: {expiry}')
        fig.show()
    else:
        print('No valid implied volatility points available.')


## Surface Heatmap


In [ ]:
if not chain.empty and {'moneyness', 'time_to_expiry', 'implied_volatility'}.issubset(chain.columns):
    valid = chain.dropna(subset=['moneyness', 'time_to_expiry', 'implied_volatility']).copy()
    if not valid.empty:
        valid['moneyness_bucket'] = pd.cut(valid['moneyness'], bins=10)
        valid['expiry_bucket'] = pd.cut(valid['time_to_expiry'], bins=8)
        surface = valid.pivot_table(index='expiry_bucket', columns='moneyness_bucket', values='implied_volatility', aggfunc='mean', observed=False)
        fig = px.imshow(surface, aspect='auto', title='IV surface heatmap')
        fig.show()
    else:
        print('No valid surface grid available.')


## Key Observations

Document where IV is unavailable, which expiries dominate activity and where public fields are insufficient.


## Limitations And Disclaimer

Black-Scholes assumptions are simplified for Russian rates, dividends and liquidity. This is research output, not advice.
